# Algoritmo Genético para Ajuste de RandomForest

Conversión de `main.py` a notebook con logs por bloque.

In [1]:
import kagglehub
import pandas as pd
import os
import random
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier

c:\Users\ASUS TUF\Documents\SEMESTRE IX\Aprendizaje máquina\algoritmos-geneticos\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# =========================================================
# 1. CARGA DE DATOS
# =========================================================
ruta = kagglehub.dataset_download("shantanudhakadd/bank-customer-churn-prediction")
print (f"Ruta del dataset: {ruta}")
archivo = [f for f in os.listdir(ruta) if f.endswith('.csv')][0]
print (f"Ruta del archivo: {archivo}")

df = pd.read_csv(os.path.join(ruta, archivo))
print(f"Columnas del Dataframe {df.columns}")
df = df.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1, errors='ignore')
df = pd.get_dummies(df, drop_first=True)
print("Primeras filas del dataset luego de categorizar columnas")
print(df.head())

X = df.drop('Exited', axis=1)
y = df['Exited']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

Ruta del dataset: C:\Users\ASUS TUF\.cache\kagglehub\datasets\shantanudhakadd\bank-customer-churn-prediction\versions\1
Ruta del archivo: Churn_Modelling.csv
Columnas del Dataframe Index(['RowNumber', 'CustomerId', 'Surname', 'CreditScore', 'Geography',
       'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard',
       'IsActiveMember', 'EstimatedSalary', 'Exited'],
      dtype='str')
Primeras filas del dataset luego de categorizar columnas
   CreditScore  Age  Tenure    Balance  NumOfProducts  HasCrCard  \
0          619   42       2       0.00              1          1   
1          608   41       1   83807.86              1          0   
2          502   42       8  159660.80              3          1   
3          699   39       1       0.00              2          0   
4          850   43       2  125510.82              1          1   

   IsActiveMember  EstimatedSalary  Exited  Geography_Germany  \
0               1        101348.88       1              False   
1

In [3]:
# =========================================================
# 2. ESPACIO DE BÚSQUEDA (HIPERPARÁMETROS)
# =========================================================
def crearIndividuo():
    """Genera una configuración aleatoria del modelo"""
    return {
        "n_estimators": random.randint(50, 300), #cantidad de árboles
        "max_depth": random.randint(5, 30),  # evitamos None (árboles muy grandes)
        "min_samples_split": random.randint(2, 10), #regla de división de nodo
        "min_samples_leaf": random.randint(1, 5), #tamaño mínimo hoja
        "max_features": random.choice(['sqrt', 'log2']), #selección de variables
        "bootstrap": random.choice([True, False]) #tipo de muestreo
    }

# =========================================================
# 3. FUNCIÓN FITNESS
# =========================================================
def evaluarFitness(ind):
    """
    Evalúa qué tan buena es una configuración.
    Usa validación cruzada (más robusto que un solo split).
    """
    modelo = RandomForestClassifier(
        **ind,
        random_state=42,
        n_jobs=-1 #todos los nucleos de la cpu
    )

    score = cross_val_score( #validacion cruzada
        modelo,
        X_train,
        y_train,
        cv=3,  # reducido para eficiencia divide en tres el datasets
        scoring='f1'
    ).mean() #devuelve el promedio de F1

    return score

In [4]:
# =========================================================
# 4. SELECCIÓN POR TORNEO (MEJOR QUE TRUNCADA)
# =========================================================
def seleccion_torneo(evaluados, k=3):
    """
    Selecciona un individuo mediante torneo.
    - Toma k individuos al azar
    - Retorna el mejor
    """
    participantes = random.sample(evaluados, k)
    
    print(f"\nTorneo con {k} participantes")
    print(f"Fitness de participantes: {[x['fitness'] for x in participantes]}")

    ganador = max(participantes, key=lambda x: x["fitness"])

    print(f"Ganador seleccionado: {ganador['fitness']}")
    
    return ganador


# =========================================================
# 5. CRUCE (UNIFORME)
# =========================================================
def cruzar(p1, p2):
    """
    Combina hiperparámetros de ambos padres.
    Cada parámetro se hereda aleatoriamente.
    """
    print("\nCruce entre padres")
    print(f"Padre 1: {p1}")
    print(f"Padre 2: {p2}")

    hijo = {}

    for key in p1:
        hijo[key] = random.choice([p1[key], p2[key]])

    print(f"Hijo generado: {hijo}")

    return hijo


# =========================================================
# 6. MUTACIÓN (CONTROLADA)
# =========================================================
def mutar(ind, prob=0.1):  # probabilidad de mutacion de 10%
    """
    Introduce cambios aleatorios pequeños.
    Probabilidad baja para no destruir buenas soluciones.
    """
    print("\nMutación aplicada")
    print(f"Individuo antes: {ind}")

    for key in ind:
        if random.random() < prob:
            nuevo = crearIndividuo()
            print(f"Parámetro mutado: {key}")
            print(f"Nuevo valor: {nuevo[key]}")
            ind[key] = nuevo[key]

    print(f"Individuo después de mutación: {ind}")

    return ind

In [5]:
# =========================================================
# 7. ALGORITMO GENÉTICO PRINCIPAL
# =========================================================
def algoritmoGenetico(poblacion_size=20, generaciones=15):

    # Crear población inicial
    poblacion = [crearIndividuo() for _ in range(poblacion_size)]

    mejor_global = None
    mejor_score = 0

    for gen in range(generaciones):

        evaluados = []

        # -----------------------------
        # Evaluación de la población
        # -----------------------------
        for ind in poblacion:
            fitness = evaluarFitness(ind)
            evaluados.append({"ind": ind, "fitness": fitness})

            # Guardar mejor global
            if fitness > mejor_score:
                mejor_score = fitness
                mejor_global = ind

        # Ordenar (solo para mostrar log)
        evaluados.sort(key=lambda x: x["fitness"], reverse=True)

        print(f"Gen {gen+1} | Mejor F1: {evaluados[0]['fitness']:.5f}")

        # -----------------------------
        # NUEVA GENERACIÓN
        # -----------------------------
        nueva_poblacion = []

        # ELITISMO: conservar el mejor SIEMPRE
        nueva_poblacion.append(mejor_global)

        # Generar nuevos individuos
        while len(nueva_poblacion) < poblacion_size:

            # Selección por torneo
            p1 = seleccion_torneo(evaluados)["ind"]
            p2 = seleccion_torneo(evaluados)["ind"]

            # Cruce
            hijo = cruzar(p1, p2)

            # Mutación
            hijo = mutar(hijo)

            nueva_poblacion.append(hijo)

        # Reemplazo generacional
        poblacion = nueva_poblacion

    # =====================================================
    # RESULTADO FINAL
    # =====================================================
    print("\n--- RESULTADO FINAL ---")
    print("Mejor F1:", mejor_score)
    print("Mejores hiperparámetros:", mejor_global)

In [6]:
# =========================================================
# EJECUCIÓN
# =========================================================
algoritmoGenetico()

Gen 1 | Mejor F1: 0.58806

Torneo con 3 participantes
Fitness de participantes: [np.float64(0.5578401436135101), np.float64(0.5822865086387762), np.float64(0.5837938419830712)]
Ganador seleccionado: 0.5837938419830712

Torneo con 3 participantes
Fitness de participantes: [np.float64(0.47121173311087444), np.float64(0.584090071602528), np.float64(0.5837938419830712)]
Ganador seleccionado: 0.584090071602528

Cruce entre padres
Padre 1: {'n_estimators': 198, 'max_depth': 29, 'min_samples_split': 9, 'min_samples_leaf': 5, 'max_features': 'sqrt', 'bootstrap': False}
Padre 2: {'n_estimators': 203, 'max_depth': 17, 'min_samples_split': 6, 'min_samples_leaf': 4, 'max_features': 'log2', 'bootstrap': True}
Hijo generado: {'n_estimators': 203, 'max_depth': 17, 'min_samples_split': 9, 'min_samples_leaf': 5, 'max_features': 'log2', 'bootstrap': True}

Mutación aplicada
Individuo antes: {'n_estimators': 203, 'max_depth': 17, 'min_samples_split': 9, 'min_samples_leaf': 5, 'max_features': 'log2', 'boo